In [10]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier



In [22]:
data = pd.read_csv("dataset.csv", delimiter = ";")

X = pd.get_dummies(data.drop(columns=["ShotType"]), drop_first=True).values

classes = np.unique(data["ShotType"].values)
class_to_idx = {c: i for i, c in enumerate(classes)}
y = np.array([class_to_idx[c] for c in data["ShotType"].values]) #map class to integers
n_classes = len(classes)

print(X)
y

[[1 1 73.83 ... False False True]
 [0 1 28.13 ... False False True]
 [0 1 51.88 ... False False True]
 ...
 [0 1 26.86 ... False False True]
 [0 1 3.42 ... False False True]
 [0 1 2.13 ... False False True]]


array([0, 3, 0, ..., 2, 0, 2], shape=(5024,))

In [12]:
data

,ShotType,Competition,PlayerType,Transition,TwoLegged,Movement,Angle,Distance
0,above head,U14,F,1,1,no,73.83,0.73
1,layup,U14,F,0,1,no,28.13,1.02
2,above head,U14,F,0,1,no,51.88,7.22
3,above head,U14,F,1,1,no,80.84,3.64
4,above head,U14,F,0,1,no,30.89,7.20
...,...,...,...,...,...,...,...,...
5019,layup,EURO,C,0,1,no,0.00,0.52
5020,above head,EURO,C,0,1,no,60.91,4.29
5021,hook shot,EURO,C,0,1,no,26.86,0.83
5022,above head,EURO,C,0,1,no,3.42,5.38


In [ ]:
def calculate_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def calc_log_loss_multi(y_true, y_proba):
    eps = 1e-15
    y_proba = np.clip(y_proba, eps, 1 - eps) #preprecim da imamo pol log(0)
    y_proba /= y_proba.sum(axis=1)[:, np.newaxis]
    
    loss = 0
    for i in range(len(y_true)):
        loss -= np.log(y_proba[i, y_true[i]])
    return loss / len(y_true)

def custom_k_fold(n_samples, n_splits = 5, seed= 67):
    np.random.seed(seed)
    indices = np.arange(n_samples)
    np.random.shuffle(indices)

    fold_sizes = np.full(n_splits, n_samples // n_splits, dtype=int)
    fold_sizes[:n_samples%n_splits] += 1 

    folds = []
    current = 0
    for fold_size in fold_sizes:
        start, stop = current, current + fold_size
        test_idx = indices[start:stop]
        train_idx = np.concatenate([indices[:start], indices[stop:]])
        folds.append((train_idx, test_idx))
        current = stop
    return folds    

In [19]:
n_outer_splits = 10
n_inner_splits = 10
param_grid = [2,4,6,8,10, None]

results = {
    'Baseline': {'acc': [], 'log_loss': []},
    'Logistic Reg': {'acc': [], 'log_loss': []},
    'DT (Train-Opt)': {'acc': [], 'log_loss': []},
    'DT (Nested CV)': {'acc': [], 'log_loss': []}
}

outer_folds = custom_k_fold(len(X), n_splits=n_outer_splits)

all_predictions = []

#evalvacija
for train_idx, test_idx in outer_folds:
    x_train, y_train = X[train_idx], y[train_idx]
    x_test, y_test = X[test_idx], y[test_idx]

    test_angles = data['Angle'].iloc[test_idx].values
    test_comps = data['Competition'].iloc[test_idx].values

    classes, class_counts = np.unique(y_train, return_counts=True)
    class_probs = class_counts / len(y_train)
    major_class = classes[np.argmax(class_counts)]
    y_pred_base = np.full(len(y_test), major_class)

    y_prob_base = np.tile(class_probs, (len(y_test), 1))

    results['Baseline']['acc'].append(calculate_accuracy(y_test, y_pred_base))
    results['Baseline']['log_loss'].append(calc_log_loss_multi(y_test, y_prob_base))

    
    # LR
    
    log_reg_model = LogisticRegression(max_iter=1000)
    log_reg_model.fit(x_train, y_train)
    
    lr_preds = log_reg_model.predict(x_test)
    lr_probs = log_reg_model.predict_proba(x_test)
    
    results['Logistic Reg']['acc'].append(calculate_accuracy(y_test, lr_preds))
    results['Logistic Reg']['log_loss'].append(calc_log_loss_multi(y_test, lr_probs))
    
    lr_prob_correct = lr_probs[np.arange(len(y_test)), y_test] #za r

    
    # DECISION TREE (TRAIN)
    
    best_train_acc = -1
    best_train_depth = None
    
    for depth in param_grid:
        dt = DecisionTreeClassifier(max_depth=depth, random_state=67)
        dt.fit(x_train, y_train)
        score = calculate_accuracy(y_train, dt.predict(x_train)) 

        if score > best_train_acc:
            best_train_acc = score
            best_train_depth = depth

    dt_train_opt = DecisionTreeClassifier(max_depth=best_train_depth, random_state=67)
    dt_train_opt.fit(x_train, y_train)
    results["DT (Train-Opt)"]["acc"].append(calculate_accuracy(y_test, dt_train_opt.predict(x_test)))
    results["DT (Train-Opt)"]["log_loss"].append(calc_log_loss_multi(y_test, dt_train_opt.predict_proba(x_test)))

    
    # DECISION TREE NESTED CV
    
    inner_folds = custom_k_fold(len(x_train), n_splits=n_inner_splits, seed=67)
    best_val_acc = -1
    best_val_depth = None
    
    for depth in param_grid:
        inner_scores = []
        for inner_train_idx, inner_test_idx in inner_folds:
            x_in_train, y_in_train = x_train[inner_train_idx], y_train[inner_train_idx]
            x_in_val, y_in_val = x_train[inner_test_idx], y_train[inner_test_idx]

            dt_inner = DecisionTreeClassifier(max_depth=depth, random_state=67)
            dt_inner.fit(x_in_train, y_in_train)
            inner_scores.append(calculate_accuracy(y_in_val, dt_inner.predict(x_in_val)))

        avg_inner_scores = np.mean(inner_scores)
        if avg_inner_scores > best_val_acc:
            best_val_acc = avg_inner_scores
            best_val_depth = depth

    df_nested = DecisionTreeClassifier(max_depth=best_val_depth, random_state=67)
    df_nested.fit(x_train, y_train)
    
    dt_preds = df_nested.predict(x_test)
    dt_probs = df_nested.predict_proba(x_test)
    
    results['DT (Nested CV)']['acc'].append(calculate_accuracy(y_test, dt_preds))
    results['DT (Nested CV)']['log_loss'].append(calc_log_loss_multi(y_test, dt_probs))
    
    dt_prob_correct = dt_probs[np.arange(len(y_test)), y_test] # za r

    

    for i in range(len(test_idx)):
        all_predictions.append({
            'Angle': test_angles[i],
            'Competition': test_comps[i],
            'Model': 'Logistic_Regression',
            'y_true': y_test[i],
            'y_pred': lr_preds[i],
            'prob_correct_class': lr_prob_correct[i]
        })
        
        all_predictions.append({
            'Angle': test_angles[i],
            'Competition': test_comps[i],
            'Model': 'Decision_Tree_NestedCV',
            'y_true': y_test[i],
            'y_pred': dt_preds[i],
            'prob_correct_class': dt_prob_correct[i]
        })

/Users/denisnedicc/Desktop/Stuff/DenisFaks/MLDS/HW2/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/denisnedicc/Desktop/Stuff/DenisFaks/MLDS/HW2/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data 

In [20]:
for model, metrics in results.items():
    print(f"{model}:")
    print(f"  Accuracy: {np.mean(metrics['acc']):.4f} +- {np.std(metrics["acc"]):.4f}")
    print(f"  Log-Loss: {np.mean(metrics['log_loss']):.4f} +-{np.std(metrics['log_loss']):.4f}\n")

Baseline:
  Accuracy: 0.6081 +- 0.0132
  Log-Loss: 1.1658 +-0.0282

Logistic Reg:
  Accuracy: 0.7361 +- 0.0245
  Log-Loss: 0.6729 +-0.0545

DT (Train-Opt):
  Accuracy: 0.7034 +- 0.0243
  Log-Loss: 10.0925 +-0.8872

DT (Nested CV):
  Accuracy: 0.7486 +- 0.0230
  Log-Loss: 1.0858 +-0.2290



In [21]:
predictions_df = pd.DataFrame(all_predictions)
predictions_df.to_csv('predictions.csv', index=False)